## Install Library

In [ ]:
# Install the YOLOv8 package
!pip install ultralytics

# Clear the output and check the GPU status
import ultralytics
ultralytics.checks()

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.7/112.6 GB disk)


## Mount with Google Drive




In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Splitting Based On Brightness

In [ ]:
import os
import shutil
import cv2
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
# Paths to YOLO test images and labels
IMAGE_DIR = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/test/images"
LABEL_DIR = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/test/labels"

# Paths for the new sorted folders
DAY_DIR = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day"
NIGHT_DIR = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night"

# Brightness Threshold (0 to 255)
# 0 is pitch black, 255 is pure white.
BRIGHTNESS_THRESHOLD = 70

def setup_directories():
    for directory in [DAY_DIR, NIGHT_DIR]:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"Created directory: {directory}")

# Reads an image and calculates its average brightness using the HSV color space.
def get_average_brightness(image_path):
    # Read the image using OpenCV
    image = cv2.imread(image_path)

    if image is None:
        print(f"Warning: Could not read {image_path}")
        return None

    # Convert the image from BGR to HSV
    # HSV separates color (Hue, Saturation) from brightness (Value)
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Extract the 'Value' (brightness) channel
    _, _, v_channel = cv2.split(hsv_image)

    # Calculate and return the mean brightness
    mean_brightness = np.mean(v_channel)
    return mean_brightness

def sort_dataset():
    print("Starting dataset sorting...")

    day_count = 0
    night_count = 0

    # Iterate through all files in the images directory
    for filename in os.listdir(IMAGE_DIR):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(IMAGE_DIR, filename)

            # 1. Calculate brightness
            brightness = get_average_brightness(image_path)
            if brightness is None:
                continue

            # 2. Determine destination folder
            if brightness >= BRIGHTNESS_THRESHOLD:
                dest_dir = DAY_DIR
                day_count += 1
            else:
                dest_dir = NIGHT_DIR
                night_count += 1

            # Create images/labels subfolders in the destination
            dest_image_dir = os.path.join(dest_dir, "images")
            dest_label_dir = os.path.join(dest_dir, "labels")
            os.makedirs(dest_image_dir, exist_ok=True)
            os.makedirs(dest_label_dir, exist_ok=True)

            # 3. Copy the image to the new folder
            dest_image_path = os.path.join(dest_image_dir, filename)
            shutil.copy2(image_path, dest_image_path)

            # 4. Check for and copy the matching YOLO .txt label file
            base_name = os.path.splitext(filename)[0]
            label_filename = base_name + ".txt"
            label_path = os.path.join(LABEL_DIR, label_filename)

            if os.path.exists(label_path):
                dest_label_path = os.path.join(dest_label_dir, label_filename)
                shutil.copy2(label_path, dest_label_path)

    print("\n==========================================")
    print("Sorting Complete!")
    print(f"Total Normal Light (Day) Images: {day_count}")
    print(f"Total Low Light (Night) Images:  {night_count}")
    print("==========================================")

if __name__ == "__main__":
    # Ensure input directory exists before running
    if not os.path.exists(IMAGE_DIR):
        print(f"Error: Image directory '{IMAGE_DIR}' not found!")
        print("Please check your folder structure.")
    else:
        setup_directories()
        sort_dataset()

Created directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day
Created directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night
Starting dataset sorting...

Sorting Complete!
Total Normal Light (Day) Images: 317
Total Low Light (Night) Images:  290


## Synchronizing Labels

In [ ]:
import os
import shutil

# 1. Define paths
original_labels_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/test/labels"

day_images_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/images"
day_labels_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/labels"

night_images_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night/images"
night_labels_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night/labels"

def sync_labels(images_dir, target_labels_dir, source_labels_dir):
    os.makedirs(target_labels_dir, exist_ok=True)

    # Get all image filenames currently in the folder
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(valid_extensions)]

    copied_count = 0
    missing_count = 0

    for img_file in image_files:
        # Get the filename without extension (e.g., "pothole_01" from "pothole_01.jpg")
        base_name = os.path.splitext(img_file)[0]
        label_name = base_name + ".txt"

        source_label_path = os.path.join(source_labels_dir, label_name)
        target_label_path = os.path.join(target_labels_dir, label_name)

        # Copy the matching .txt file if it exists
        if os.path.exists(source_label_path):
            shutil.copy2(source_label_path, target_label_path)
            copied_count += 1
        else:
            print(f"⚠️ Warning: No label file found for {img_file}")
            missing_count += 1

    return len(image_files), copied_count, missing_count

print("🔄 Synchronizing Day Labels...")
day_imgs, day_lbls, day_miss = sync_labels(day_images_dir, day_labels_dir, original_labels_dir)

print("🔄 Synchronizing Night Labels...")
night_imgs, night_lbls, night_miss = sync_labels(night_images_dir, night_labels_dir, original_labels_dir)

print("\n==========================================")
print(f"☀️ Normal Light (Day):   {day_imgs} images | {day_lbls} labels matched")
print(f"🌙 Low Light (Night):    {night_imgs} images | {night_lbls} labels matched")
print("==========================================")
if day_miss > 0 or night_miss > 0:
    print(f"⚠️ Note: {day_miss + night_miss} images were missing their .txt label files in the original folder.")
else:
    print("✅ 100% Perfect Match! Your manual split is ready for YOLO evaluation.")

🔄 Synchronizing Day Labels...
🔄 Synchronizing Night Labels...

☀️ Normal Light (Day):   294 images | 294 labels matched
🌙 Low Light (Night):    289 images | 289 labels matched
✅ 100% Perfect Match! Your manual split is ready for YOLO evaluation.


## Dataset Balancing

In [ ]:
import os
import shutil

# 1. Define paths
day_images_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/images"
day_labels_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/labels"

# Create a backup folder for the removed images
backup_images_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day_excluded/images"
backup_labels_dir = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day_excluded/labels"
os.makedirs(backup_images_dir, exist_ok=True)
os.makedirs(backup_labels_dir, exist_ok=True)

# 2. Count the number of potholes (lines) in each label file
image_pothole_counts = []

for label_file in os.listdir(day_labels_dir):
    if label_file.endswith(".txt"):
        label_path = os.path.join(day_labels_dir, label_file)
        with open(label_path, 'r') as f:
            lines = [line for line in f.readlines() if line.strip()]
            pothole_count = len(lines)

        base_name = os.path.splitext(label_file)[0]
        image_pothole_counts.append((base_name, pothole_count))

# 3. Sort images from Most potholes to Least potholes
image_pothole_counts.sort(key=lambda x: x[1], reverse=True)

# 4. Target: Move the top 25 most crowded images to the backup folder
images_to_remove = 25
removed_instances = 0

print(f"🧹 Moving the top {images_to_remove} most crowded images out of test_day...\n")

for i in range(images_to_remove):
    base_name, count = image_pothole_counts[i]

    # Find image file (check .jpg, .jpeg, .png)
    for ext in ['.jpg', '.jpeg', '.png']:
        img_name = base_name + ext
        img_path = os.path.join(day_images_dir, img_name)
        if os.path.exists(img_path):
            shutil.move(img_path, os.path.join(backup_images_dir, img_name))
            break

    # Move matching label file
    lbl_path = os.path.join(day_labels_dir, base_name + ".txt")
    if os.path.exists(lbl_path):
        shutil.move(lbl_path, os.path.join(backup_labels_dir, base_name + ".txt"))

    removed_instances += count
    print(f"Moved: {base_name} (contained {count} potholes)")

# 5. Calculate new totals
remaining_images = len(os.listdir(day_images_dir))
remaining_instances = sum(count for _, count in image_pothole_counts[images_to_remove:])

print("\n==========================================")
print("✅ Dataset Balancing Complete!")
print(f"New Day Images Count:    {remaining_images} (Target was ~289)")
print(f"New Day Potholes Count:  {remaining_instances} (Target was ~571)")
print("==========================================")

🧹 Moving the top 25 most crowded images out of test_day...

Moved: pothole_325_jpg.rf.4cafa9ff2a9a7fcf3b9d9259913fb510 (contained 6 potholes)
Moved: pothole_691_jpg.rf.f7b506b89ae1db12b0ac23e34ebdec5f (contained 5 potholes)
Moved: pothole_1658_jpg.rf.734d27108f76988e1b95d750f0e547a9 (contained 5 potholes)
Moved: img-237_jpg.rf.17a7b46a40ba9580ad1aba961470aa43 (contained 5 potholes)
Moved: pothole_241_jpg.rf.a3b5b90e8a339c5eec3f1464f11cd208 (contained 5 potholes)
Moved: pothole_314_jpg.rf.5968e56951fa91c0aa884b72b8733879 (contained 5 potholes)
Moved: pothole_170_jpg.rf.019a6d0e886b26c4ffab8d39c7c528c7 (contained 5 potholes)
Moved: pothole_898_jpg.rf.ead91031d572449fffe2bc163cd71cc7 (contained 5 potholes)
Moved: potholes270_png.rf.5d1c811921020d66b5811b908281ab44 (contained 5 potholes)
Moved: pothole_1757_jpg.rf.da4f820592ad2bd7d7ce44aeb27a7556 (contained 4 potholes)
Moved: img-240_jpg.rf.d2ebe48c87dc1cf712ecef5e877c33bc (contained 4 potholes)
Moved: potholes153_png.rf.9ba53874005664add9